# Ames Housing Price Prediction — Machine Learning Models

## Load Packages

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
import tensorflow as tf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='darkgrid')

TARGET = 'logSalePrice'

## Load Data

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
df_train.head()

## Feature Engineering

In [ ]:
# Mark origin so we can split back later, then combine for consistent preprocessing
df_train['is_train'] = 1
df_test['is_train']  = 0
df_test['SalePrice'] = np.nan

df_all = pd.concat([df_train, df_test], ignore_index=True)

# Fill columns that feed into derived features (NaN = no structure, treat as 0 or use YearBuilt)
df_all['BsmtFinSF1']   = df_all['BsmtFinSF1'].fillna(0)
df_all['BsmtFinSF2']   = df_all['BsmtFinSF2'].fillna(0)
df_all['BsmtFullBath'] = df_all['BsmtFullBath'].fillna(0)
df_all['BsmtHalfBath'] = df_all['BsmtHalfBath'].fillna(0)
df_all['GarageYrBlt']  = df_all['GarageYrBlt'].fillna(df_all['YearBuilt'])
df_all['GarageArea']   = df_all['GarageArea'].fillna(0)
df_all['GarageCars']   = df_all['GarageCars'].fillna(0)
df_all['MasVnrArea']   = df_all['MasVnrArea'].fillna(0)

# Derived features
df_all['TotalSqftCalc']  = df_all['BsmtFinSF1'] + df_all['BsmtFinSF2'] + df_all['GrLivArea']
df_all['TotalFloorSF']   = df_all['1stFlrSF'] + df_all['2ndFlrSF']
df_all['HouseAge']       = df_all['YrSold'] - df_all['YearBuilt']
df_all['RemodAge']       = df_all['YrSold'] - df_all['YearRemodAdd']
df_all['GarageAge']      = df_all['YrSold'] - df_all['GarageYrBlt']
df_all['TotalBath']      = (df_all['FullBath'] + df_all['HalfBath'] * 0.5
                            + df_all['BsmtFullBath'] + df_all['BsmtHalfBath'] * 0.5)
df_all['QualityIndex']   = df_all['OverallQual'] * df_all['OverallCond']
df_all['TotalOutdoorSF'] = (df_all['WoodDeckSF'] + df_all['OpenPorchSF']
                            + df_all['EnclosedPorch'] + df_all['ScreenPorch'])

# Log-transform target (NaN for test rows — kept separate, never used as features)
df_all['logSalePrice']     = np.log(df_all['SalePrice'])
df_all['logTotalSqftCalc'] = np.log(df_all['TotalSqftCalc'])

df_all[['TotalSqftCalc','TotalFloorSF','HouseAge','TotalBath','QualityIndex']].describe()

In [ ]:
# Columns to exclude from all preprocessing loops
SKIP = ['SalePrice', 'logSalePrice', 'is_train', 'Id']

# Missing value imputation — numeric columns
# Creates M_ (missingness flag) and IMP_ (median-imputed) versions; drops original
dt = df_all.dtypes
numList = [i for i in dt.index if i not in SKIP and dt[i] in ['float64','int64']]

for i in numList:
    if df_all[i].isna().sum() == 0:
        continue
    df_all['M_'   + i] = df_all[i].isna().astype(int)
    df_all['IMP_' + i] = df_all[i].fillna(df_all[i].median())
    df_all = df_all.drop(i, axis=1)

print('Imputation complete. Shape:', df_all.shape)

In [ ]:
# Categorical encoding — fill NaN with 'MISSING', then one-hot encode with z_ prefix
dt = df_all.dtypes
objList = [i for i in dt.index if i not in SKIP and dt[i] == 'object']

for i in objList:
    df_all[i] = df_all[i].fillna('MISSING')

for i in objList:
    dummies = pd.get_dummies(df_all[i], dtype=int, prefix='z_' + i, dummy_na=False)
    df_all = pd.concat([df_all, dummies], axis=1)
    df_all = df_all.drop(i, axis=1)

print('Encoding complete. Shape:', df_all.shape)

In [ ]:
# Outlier handling — truncate at mean + 3 SD, flag with O_, rename to TRUNC_
dt = df_all.dtypes
numList = [i for i in dt.index if i not in SKIP and dt[i] in ['float64','int64']]

for i in numList:
    theMean   = df_all[i].mean()
    theSD     = df_all[i].std()
    theCutoff = round(theMean + 3 * theSD)
    if df_all[i].max() <= theCutoff:
        continue
    df_all['O_'     + i] = (df_all[i] > theCutoff).astype(int)
    df_all['TRUNC_' + i] = df_all[i].clip(upper=theCutoff)
    df_all = df_all.drop(i, axis=1)

print('Outlier handling complete. Shape:', df_all.shape)

In [ ]:
# Split back into processed train and test sets
train_processed = df_all[df_all['is_train'] == 1].drop('is_train', axis=1).copy()
test_processed  = df_all[df_all['is_train'] == 0].drop('is_train', axis=1).copy()

print('Train processed:', train_processed.shape)
print('Test processed: ', test_processed.shape)

# Confirm no remaining missing in train features
feat_cols = [c for c in train_processed.columns if c not in SKIP]
remaining_missing = train_processed[feat_cols].isnull().sum().sum()
print(f'Remaining missing values in train features: {remaining_missing}')

## EDA

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=train_processed, x='logSalePrice', kde=True, ax=axs[0])
axs[0].set_title('Distribution of logSalePrice')

num_cols = train_processed.select_dtypes(include=[np.number]).columns
drop_for_corr = ['SalePrice', 'logSalePrice']
corr_cols = [c for c in num_cols if c not in drop_for_corr]
corr = train_processed[corr_cols + ['logSalePrice']].corr()['logSalePrice'].drop('logSalePrice').sort_values()
top = pd.concat([corr.head(8), corr.tail(8)])

sns.barplot(x=top.values, y=top.index,
            palette=['salmon' if v < 0 else 'steelblue' for v in top.values], ax=axs[1])
axs[1].set_title('Top Correlations with logSalePrice')
axs[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='TRUNC_OverallQual', y='logSalePrice', data=train_processed, ax=axs[0])
axs[0].set_title('logSalePrice by Overall Quality')

sns.scatterplot(x='logTotalSqftCalc', y='logSalePrice', data=train_processed,
                alpha=0.3, ax=axs[1])
axs[1].set_title('logSalePrice vs logTotalSqftCalc')

plt.tight_layout()
plt.show()

## Models

In [ ]:
def getAmtAccuracyScores(NAME, MODEL, X, Y):
    pred = MODEL.predict(X)
    if hasattr(pred, 'flatten'):
        pred = pred.flatten()
    RMSE = math.sqrt(metrics.mean_squared_error(Y, pred))
    return [NAME, RMSE]

def getEnsembleTreeVars(ENSTREE, varNames):
    importance = ENSTREE.feature_importances_
    avg_imp = np.average(importance)
    theList = []
    for i, imp_val in enumerate(importance):
        if imp_val > avg_imp:
            v = int(imp_val / np.max(importance) * 100)
            theList.append((varNames[i], v))
    return sorted(theList, key=lambda x: x[1], reverse=True)

In [ ]:
# Feature matrix and target — drop ID and both price columns from X
drop_cols = [c for c in ['SalePrice', 'logSalePrice', 'Id'] if c in train_processed.columns]

X = train_processed.drop(drop_cols, axis=1)
Y = train_processed[TARGET]

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=5)

feature_cols = list(X.columns)
print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')

### Decision Tree

In [ ]:
WHO = 'TREE'

TREE_MODEL = tree.DecisionTreeRegressor(max_depth=6, random_state=5)
TREE_MODEL.fit(X_train, Y_train)

TRAIN_TREE = getAmtAccuracyScores(WHO + '_Train', TREE_MODEL, X_train, Y_train)
VAL_TREE   = getAmtAccuracyScores(WHO, TREE_MODEL, X_val, Y_val)

print(f'{WHO} — Train RMSE: {TRAIN_TREE[1]:.4f}  |  Val RMSE: {VAL_TREE[1]:.4f}')

vars_tree = [v[0] for v in getEnsembleTreeVars(TREE_MODEL, feature_cols)]

### Random Forest

In [ ]:
WHO = 'RF'

RF_MODEL = RandomForestRegressor(n_estimators=100, random_state=5)
RF_MODEL.fit(X_train, Y_train)

TRAIN_RF = getAmtAccuracyScores(WHO + '_Train', RF_MODEL, X_train, Y_train)
VAL_RF   = getAmtAccuracyScores(WHO, RF_MODEL, X_val, Y_val)

print(f'{WHO} — Train RMSE: {TRAIN_RF[1]:.4f}  |  Val RMSE: {VAL_RF[1]:.4f}')

RF_features = [v[0] for v in getEnsembleTreeVars(RF_MODEL, feature_cols)]
print(f'RF selected {len(RF_features)} features above average importance')

### Gradient Boosting

In [ ]:
WHO = 'GB'

GB_MODEL = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, random_state=5)
GB_MODEL.fit(X_train, Y_train)

TRAIN_GB = getAmtAccuracyScores(WHO + '_Train', GB_MODEL, X_train, Y_train)
VAL_GB   = getAmtAccuracyScores(WHO, GB_MODEL, X_val, Y_val)

print(f'{WHO} — Train RMSE: {TRAIN_GB[1]:.4f}  |  Val RMSE: {VAL_GB[1]:.4f}')

GB_features = [v[0] for v in getEnsembleTreeVars(GB_MODEL, feature_cols)]
print(f'GB selected {len(GB_features)} features above average importance')
for v in getEnsembleTreeVars(GB_MODEL, feature_cols)[:10]:
    print(v)

### Stepwise Regression

In [ ]:
# Use GB important features as the pool for forward selection
V_train    = X_train[GB_features]
stepNames  = list(V_train.columns)
maxCols    = V_train.shape[1]

sfs = SFS(LinearRegression(),
          k_features=(1, maxCols),
          forward=True,
          floating=False,
          scoring='r2',
          cv=5)
sfs.fit(V_train.values, Y_train.values)

theFigure = plot_sfs(sfs.get_metric_dict(), kind=None)
plt.title('Stepwise Forward Selection — R² by Number of Features')
plt.grid()
plt.show()

# Find the feature subset with the best cross-validated R²
dfm = pd.DataFrame.from_dict(sfs.get_metric_dict()).T
dfm = dfm[['feature_names', 'avg_score']]
dfm.avg_score = dfm.avg_score.astype(float)

maxIndex = dfm.avg_score.argmax()
stepVars = dfm.iloc[maxIndex]['feature_names']

finalStepVars = []
for i in stepVars:
    try:
        finalStepVars.append(stepNames[int(i)])
    except:
        pass

print(f'Stepwise selected {len(finalStepVars)} features:')
for v in finalStepVars:
    print(' ', v)

In [ ]:
WHO = 'STEP'

STEP_MODEL = LinearRegression()
STEP_MODEL.fit(X_train[finalStepVars], Y_train)

TRAIN_STEP = getAmtAccuracyScores(WHO + '_Train', STEP_MODEL, X_train[finalStepVars], Y_train)
VAL_STEP   = getAmtAccuracyScores(WHO, STEP_MODEL, X_val[finalStepVars], Y_val)

print(f'{WHO} — Train RMSE: {TRAIN_STEP[1]:.4f}  |  Val RMSE: {VAL_STEP[1]:.4f}')

### TensorFlow

In [ ]:
WHO = 'TF'

# Scale to [0,1] using training data — apply same scaler to val and test
theScaler = MinMaxScaler()
theScaler.fit(X_train[GB_features])

TF_Xtrain = theScaler.transform(X_train[GB_features])
TF_Xval   = theScaler.transform(X_val[GB_features])

theShapeSize = TF_Xtrain.shape[1]
theUnits     = int(2 * theShapeSize / 3)

TF_MODEL = tf.keras.Sequential([
    tf.keras.layers.Dense(theUnits, activation='relu', input_dim=theShapeSize),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(theUnits, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])

TF_MODEL.compile(loss='mse', optimizer='adam')
TF_MODEL.fit(TF_Xtrain, Y_train, epochs=100, verbose=False)

train_rmse_tf = math.sqrt(metrics.mean_squared_error(Y_train, TF_MODEL.predict(TF_Xtrain).flatten()))
val_rmse_tf   = math.sqrt(metrics.mean_squared_error(Y_val,   TF_MODEL.predict(TF_Xval).flatten()))

TRAIN_TF = [WHO + '_Train', train_rmse_tf]
VAL_TF   = [WHO, val_rmse_tf]

print(f'{WHO} — Train RMSE: {train_rmse_tf:.4f}  |  Val RMSE: {val_rmse_tf:.4f}')

### Model Comparison

In [ ]:
ALL_RESULTS = sorted(
    [VAL_TREE, VAL_RF, VAL_GB, VAL_STEP, VAL_TF],
    key=lambda x: x[1]
)

print('Model Comparison — Validation RMSE (log scale, lower is better)')
print('=' * 45)
for result in ALL_RESULTS:
    print(f'{result[0]:<10}  RMSE: {result[1]:.4f}')

BEST_NAME = ALL_RESULTS[0][0]
print(f'\nBest Model: {BEST_NAME}')

## Test Prediction

In [ ]:
# Align test features to match training columns exactly
drop_cols = [c for c in ['SalePrice', 'logSalePrice', 'Id'] if c in test_processed.columns]
X_test_kaggle = test_processed.drop(drop_cols, axis=1)

# Ensure same column order as training
X_test_kaggle = X_test_kaggle[feature_cols]

# Predict with each model — back-transform from log scale
preds = {}

preds['TREE'] = np.exp(TREE_MODEL.predict(X_test_kaggle))
preds['RF']   = np.exp(RF_MODEL.predict(X_test_kaggle))
preds['GB']   = np.exp(GB_MODEL.predict(X_test_kaggle))
preds['STEP'] = np.exp(STEP_MODEL.predict(X_test_kaggle[finalStepVars]))

TF_Xtest       = theScaler.transform(X_test_kaggle[GB_features])
preds['TF']    = np.exp(TF_MODEL.predict(TF_Xtest).flatten())

# Use best model from comparison
best_preds = preds[BEST_NAME]

submission = test_processed[['Id']].copy()
submission['SalePrice'] = best_preds

print(f'Using predictions from: {BEST_NAME}')
print(submission['SalePrice'].describe())
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)
print('submission.csv saved.')